# Modular PDE-Constrained Optimization with Neural Networks

This notebook provides a unified interface to run all examples from the paper using the modular components.

## Google Colab GPU Setup (Optional)

To run this with GPU acceleration in Google Colab:
1. Upload this notebook to Google Colab
2. Go to Runtime → Change runtime type → Select GPU/TPU
3. Run the setup cells below

### For VSCode Integration with Colab GPU:
1. Get ngrok token from https://dashboard.ngrok.com/auth/your-authtoken
2. Run the SSH setup cell below (uncomment first)
3. Connect VSCode using Remote-SSH extension

In [ ]:
# Optional: Setup for Google Colab with GPU
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    # Mount Google Drive for persistent storage
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Install required packages
    !pip install -q jax[cuda] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax matplotlib
    
    # Check GPU availability
    !nvidia-smi
    
    # Optional: SSH setup for VSCode (uncomment to use)
    # !pip install colab_ssh --upgrade
    # ngrokToken = 'YOUR_NGROK_TOKEN_HERE'
    # password = 'YOUR_PASSWORD_HERE'
    # from colab_ssh import launch_ssh
    # launch_ssh(ngrokToken, password)
else:
    print("Running locally")
    
# Check JAX devices
import jax
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

## Setup and Imports

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import matplotlib.pyplot as plt
import numpy as np
from functools import partial
from typing import Dict, Any, Tuple, Optional
import time

# If modules don't exist yet, we'll define them inline
try:
    from solvers import get_solver, HeatEquationFD, HeatEquationFEM, HeatEquationCrankNicolson
    from problems import get_problem
    from examples import get_example, create_neural_network
    print("Modules loaded successfully")
except ImportError:
    print("Warning: Modules not found. Copy the module files to Colab or define inline.")
    # You can paste the module code here if needed

# Set random seed for reproducibility
key = jax.random.PRNGKey(42)

# Configure matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## Module Definitions (if not imported)

If the modules aren't available, uncomment and run the cells below:

In [ ]:
# Optional: Define create_neural_network function if modules not available
def create_neural_network(hidden_layers: list = [256, 256], activation: str = 'tanh'):
    """Create a neural network for force/parameter approximation."""
    
    class Network(nn.Module):
        layers: list
        activation: str
        
        @nn.compact
        def __call__(self, x):
            for i, features in enumerate(self.layers):
                x = nn.Dense(features)(x)
                if i < len(self.layers) - 1:
                    if self.activation == 'tanh':
                        x = nn.tanh(x)
                    elif self.activation == 'relu':
                        x = nn.relu(x)
            x = nn.Dense(1)(x)  # Output layer
            return x.squeeze(-1)
    
    return Network(layers=hidden_layers, activation=activation)

## Configuration: Select Problem and Solver

In [ ]:
# Configuration - modify these to run different examples
CONFIG = {
    # Problem selection
    'problem': 'heat-1d',  # Options: 'poisson-1d-scalar', 'poisson-1d-vector', 'heat-1d', 'wave-1d', etc.
    'problem_params': {'T': 1.0, 'zero_ic': True},
    
    # Solver selection  
    'solver': 'heat',  # Options: 'heat', 'poisson', 'wave', 'advection-diffusion'
    'discretization': 'fem',  # Options: 'fd', 'fem', 'crank-nicolson'
    
    # Grid parameters
    'nx': 32,  # Spatial grid points
    'nt': 32,  # Temporal grid points (for time-dependent problems)
    'L': 1.0,  # Spatial domain length
    'T': 1.0,  # Temporal domain length
    
    # Optimization settings
    'optimization_type': 'force',  # Options: 'force', 'initial_condition', 'parameter'
    'use_neural_network': True,  # Use NN for force approximation
    'nn_layers': [256, 256],  # Neural network architecture
    'nn_activation': 'tanh',  # Activation function
    
    # Training parameters
    'learning_rate': 1e-3,
    'optimizer': 'adamw',  # Options: 'adam', 'adamw', 'sgd', 'rmsprop'
    'max_iterations': 2000,
    'regularization': 1e-6,
    'log_interval': 200,  # Print loss every N iterations
    
    # Plotting
    'plot_results': True,
    'save_plots': False,
    'plot_dir': './plots/'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Initialize Problem and Solver

In [ ]:
# Initialize problem
try:
    problem = get_problem(CONFIG['problem'], **CONFIG['problem_params'])
    print(f"\nProblem: {problem.name}")
    print(f"Description: {problem.description}")
    print(f"Domain: {problem.domain}")
    print(f"Boundary conditions: {problem.boundary_conditions}")
except:
    print("Define problem manually - see examples in problems.py")

# Initialize solver
try:
    if CONFIG['solver'] == 'poisson' and '2d' not in CONFIG['problem']:
        # 1D Poisson doesn't need nt
        solver = get_solver(CONFIG['solver'], CONFIG['discretization'], 
                          nx=CONFIG['nx'], ny=CONFIG['nx'])
    else:
        solver = get_solver(CONFIG['solver'], CONFIG['discretization'],
                          nx=CONFIG['nx'], nt=CONFIG['nt'],
                          L=CONFIG['L'], T=CONFIG['T'])
    print(f"\nSolver initialized: {CONFIG['solver']} with {CONFIG['discretization']}")
    print(f"Grid: nx={CONFIG['nx']}, nt={CONFIG['nt']}")
except Exception as e:
    print(f"Error initializing solver: {e}")
    print("Define solver manually - see examples in solvers.py")

## Setup Target Solution and System Matrix

In [ ]:
# Create system matrix
print("Creating system matrix...")
A_system = solver.create_system_matrix()
print(f"System matrix shape: {A_system.shape}")
print(f"Matrix condition number: {jnp.linalg.cond(A_system):.2e}")

# Setup target solution
x_grid = solver.x_grid
if hasattr(solver, 't_grid'):
    t_grid = solver.t_grid
    if hasattr(problem, 'analytical_solution'):
        u_target = problem.analytical_solution(x_grid, t_grid)
    else:
        # Default target for testing
        X, T = jnp.meshgrid(x_grid, t_grid, indexing='ij')
        u_target = jnp.sin(jnp.pi * X) * jnp.sin(jnp.pi * T)
else:
    if hasattr(problem, 'analytical_solution'):
        u_target = problem.analytical_solution(x_grid)
    else:
        u_target = jnp.sin(jnp.pi * x_grid)

u_target_vec = u_target.flatten()
print(f"Target solution shape: {u_target.shape}")
print(f"Vectorized target shape: {u_target_vec.shape}")

## Setup Neural Network (if enabled)

In [ ]:
if CONFIG['use_neural_network']:
    print("Setting up neural network...")
    
    # Create input coordinates
    if hasattr(solver, 't_grid'):
        coords = []
        for i in range(len(x_grid)):
            for j in range(len(t_grid)):
                coords.append([x_grid[i], t_grid[j]])
        input_coords = jnp.array(coords)
    else:
        input_coords = x_grid.reshape(-1, 1)
    
    print(f"Input coordinates shape: {input_coords.shape}")
    
    # Initialize network
    model = create_neural_network(CONFIG['nn_layers'], CONFIG['nn_activation'])
    key = jax.random.PRNGKey(42)
    params = model.init(key, input_coords)
    
    # Count parameters
    n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    print(f"Neural network initialized with {n_params} parameters")
    print(f"Architecture: {CONFIG['nn_layers']} with {CONFIG['nn_activation']} activation")
else:
    print("Direct force optimization (no neural network)")
    # Initialize force vector directly
    params = jnp.zeros(len(u_target_vec))

## Define Loss Function

In [ ]:
@jax.jit
def loss_fn(params):
    """Compute loss function for optimization."""
    
    if CONFIG['use_neural_network']:
        # Get force from neural network
        force_pred = model.apply(params, input_coords)
    else:
        # Direct force optimization
        force_pred = params
    
    # Solve PDE: A*u = f
    u_pred = jnp.linalg.solve(A_system, force_pred)
    
    # Compute losses
    data_loss = jnp.mean((u_pred - u_target_vec)**2)
    reg_loss = CONFIG['regularization'] * jnp.mean(force_pred**2)
    total_loss = data_loss + reg_loss
    
    return total_loss, (data_loss, reg_loss, u_pred, force_pred)

# Test loss function
test_loss, (test_data, test_reg, test_u, test_f) = loss_fn(params)
print(f"\nInitial loss: {test_loss:.6f}")
print(f"  Data loss: {test_data:.6f}")
print(f"  Regularization loss: {test_reg:.6f}")

## Setup Optimizer

In [ ]:
# Select optimizer
optimizer_map = {
    'adam': optax.adam,
    'adamw': optax.adamw,
    'sgd': optax.sgd,
    'rmsprop': optax.rmsprop,
}

if CONFIG['optimizer'] in optimizer_map:
    optimizer = optimizer_map[CONFIG['optimizer']](CONFIG['learning_rate'])
else:
    print(f"Unknown optimizer {CONFIG['optimizer']}, using Adam")
    optimizer = optax.adam(CONFIG['learning_rate'])

opt_state = optimizer.init(params)
print(f"Optimizer: {CONFIG['optimizer']} with learning rate {CONFIG['learning_rate']}")

## Training Loop

In [ ]:
@jax.jit
def train_step(params, opt_state):
    """Single training step."""
    (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux

# Training history
history = {
    'loss': [],
    'data_loss': [],
    'reg_loss': [],
    'time': [],
}

print(f"\nStarting training for {CONFIG['max_iterations']} iterations...")
print("="*60)

start_time = time.time()

for iteration in range(CONFIG['max_iterations']):
    params, opt_state, loss, (data_loss, reg_loss, u_pred, force_pred) = train_step(params, opt_state)
    
    # Store history
    history['loss'].append(float(loss))
    history['data_loss'].append(float(data_loss))
    history['reg_loss'].append(float(reg_loss))
    history['time'].append(time.time() - start_time)
    
    # Logging
    if iteration % CONFIG['log_interval'] == 0:
        elapsed = time.time() - start_time
        print(f"Iter {iteration:4d} | Time: {elapsed:6.2f}s | Loss: {loss:.6e} | "
              f"Data: {data_loss:.6e} | Reg: {reg_loss:.6e}")
    
    # Early stopping
    if loss < 1e-10:
        print(f"\nConverged at iteration {iteration} with loss {loss:.2e}")
        break

# Final results
final_time = time.time() - start_time
print("="*60)
print(f"\nTraining completed in {final_time:.2f} seconds")
print(f"Final loss: {history['loss'][-1]:.6e}")
print(f"Final data loss: {history['data_loss'][-1]:.6e}")
print(f"Final regularization loss: {history['reg_loss'][-1]:.6e}")

# Get final predictions
if CONFIG['use_neural_network']:
    force_final = model.apply(params, input_coords)
else:
    force_final = params
u_final = jnp.linalg.solve(A_system, force_final)

## Visualization: Loss History

In [ ]:
if CONFIG['plot_results']:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss over iterations
    axes[0].semilogy(history['loss'], 'b-', alpha=0.7, label='Total Loss')
    axes[0].semilogy(history['data_loss'], 'r-', alpha=0.7, label='Data Loss')
    axes[0].semilogy(history['reg_loss'], 'g-', alpha=0.7, label='Reg Loss')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss History')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss over time
    axes[1].semilogy(history['time'], history['loss'], 'b-', alpha=0.7, label='Total Loss')
    axes[1].semilogy(history['time'], history['data_loss'], 'r-', alpha=0.7, label='Data Loss')
    axes[1].set_xlabel('Time (seconds)')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss vs Wall Time')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if CONFIG['save_plots']:
        plt.savefig(f"{CONFIG['plot_dir']}/loss_history.png", dpi=150)
    plt.show()

## Visualization: Solutions

In [ ]:
if CONFIG['plot_results'] and hasattr(solver, 't_grid'):
    # Reshape for 2D plotting
    nx, nt = CONFIG['nx'], CONFIG['nt']
    u_target_2d = u_target.reshape(nx, nt)
    u_final_2d = u_final.reshape(nx, nt)
    force_2d = force_final.reshape(nx, nt)
    error_2d = u_final_2d - u_target_2d
    
    # Create meshgrid for plotting
    X, T = jnp.meshgrid(x_grid, t_grid, indexing='ij')
    
    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Target solution
    im1 = axes[0,0].contourf(T, X, u_target_2d, levels=20, cmap='viridis')
    axes[0,0].set_xlabel('Time t')
    axes[0,0].set_ylabel('Space x')
    axes[0,0].set_title('Target Solution')
    plt.colorbar(im1, ax=axes[0,0])
    
    # Predicted solution
    im2 = axes[0,1].contourf(T, X, u_final_2d, levels=20, cmap='viridis')
    axes[0,1].set_xlabel('Time t')
    axes[0,1].set_ylabel('Space x')
    axes[0,1].set_title('Predicted Solution')
    plt.colorbar(im2, ax=axes[0,1])
    
    # Error
    im3 = axes[1,0].contourf(T, X, error_2d, levels=20, cmap='RdBu_r')
    axes[1,0].set_xlabel('Time t')
    axes[1,0].set_ylabel('Space x')
    axes[1,0].set_title('Error: Predicted - Target')
    plt.colorbar(im3, ax=axes[1,0])
    
    # Learned force
    im4 = axes[1,1].contourf(T, X, force_2d, levels=20, cmap='plasma')
    axes[1,1].set_xlabel('Time t')
    axes[1,1].set_ylabel('Space x')
    title = 'Learned Force (NN)' if CONFIG['use_neural_network'] else 'Learned Force'
    axes[1,1].set_title(title)
    plt.colorbar(im4, ax=axes[1,1])
    
    plt.suptitle(f"{CONFIG['problem']} with {CONFIG['discretization']} discretization", fontsize=14)
    plt.tight_layout()
    
    if CONFIG['save_plots']:
        plt.savefig(f"{CONFIG['plot_dir']}/solutions.png", dpi=150)
    plt.show()
    
elif CONFIG['plot_results']:
    # 1D plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Solutions
    axes[0].plot(x_grid, u_target, 'b-', label='Target', linewidth=2)
    axes[0].plot(x_grid, u_final, 'r--', label='Predicted', linewidth=2)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('u(x)')
    axes[0].set_title('Solutions')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Error
    axes[1].plot(x_grid, u_final - u_target, 'g-', linewidth=2)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('Error')
    axes[1].set_title('Prediction Error')
    axes[1].grid(True, alpha=0.3)
    
    # Force
    axes[2].plot(x_grid, force_final, 'm-', linewidth=2)
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('f(x)')
    title = 'Learned Force (NN)' if CONFIG['use_neural_network'] else 'Learned Force'
    axes[2].set_title(title)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if CONFIG['save_plots']:
        plt.savefig(f"{CONFIG['plot_dir']}/solutions_1d.png", dpi=150)
    plt.show()

## Error Metrics

In [ ]:
# Compute error metrics
mse_solution = jnp.mean((u_final - u_target_vec)**2)
mae_solution = jnp.mean(jnp.abs(u_final - u_target_vec))
max_error = jnp.max(jnp.abs(u_final - u_target_vec))
relative_error = jnp.linalg.norm(u_final - u_target_vec) / jnp.linalg.norm(u_target_vec)

print("\nError Metrics:")
print("="*40)
print(f"MSE (Solution):      {mse_solution:.6e}")
print(f"MAE (Solution):      {mae_solution:.6e}")
print(f"Max Error:           {max_error:.6e}")
print(f"Relative L2 Error:   {relative_error:.6e}")

# If analytical force is available, compute force error
if hasattr(problem, 'source_term'):
    if hasattr(solver, 't_grid'):
        force_true = problem.source_term(x_grid, t_grid).flatten()
    else:
        force_true = problem.source_term(x_grid).flatten()
    
    if force_true is not None and len(force_true) == len(force_final):
        mse_force = jnp.mean((force_final - force_true)**2)
        print(f"MSE (Force):         {mse_force:.6e}")

## Run Multiple Configurations (Batch Testing)

In [ ]:
# Define multiple configurations to test
test_configs = [
    {
        'name': 'Heat FD',
        'solver': 'heat',
        'discretization': 'fd',
        'learning_rate': 1e-3,
    },
    {
        'name': 'Heat FEM',
        'solver': 'heat',
        'discretization': 'fem',
        'learning_rate': 1e-3,
    },
    {
        'name': 'Heat Crank-Nicolson',
        'solver': 'heat',
        'discretization': 'crank-nicolson',
        'learning_rate': 1e-3,
    },
]

# Run quick comparison (fewer iterations)
results_comparison = []

for test_config in test_configs:
    print(f"\nTesting: {test_config['name']}")
    print("-"*40)
    
    # Update config
    CONFIG.update(test_config)
    CONFIG['max_iterations'] = 100  # Quick test
    
    try:
        # Initialize solver
        solver = get_solver(CONFIG['solver'], CONFIG['discretization'],
                          nx=CONFIG['nx'], nt=CONFIG['nt'])
        
        # Setup (simplified)
        A_system = solver.create_system_matrix()
        
        # Record result
        results_comparison.append({
            'name': test_config['name'],
            'condition_number': float(jnp.linalg.cond(A_system)),
            'matrix_size': A_system.shape[0],
        })
        
        print(f"  Matrix size: {A_system.shape}")
        print(f"  Condition number: {jnp.linalg.cond(A_system):.2e}")
        
    except Exception as e:
        print(f"  Error: {e}")

# Summary
print("\n" + "="*60)
print("Comparison Summary:")
print("="*60)
for result in results_comparison:
    print(f"{result['name']:20s} | Size: {result['matrix_size']:5d} | Cond: {result['condition_number']:.2e}")

## Save Results

In [ ]:
# Save results to file (optional)
if IN_COLAB:
    # Save to Google Drive if in Colab
    save_path = '/content/drive/MyDrive/pde_results/'
else:
    save_path = './results/'

# Uncomment to save
# import pickle
# import os
# os.makedirs(save_path, exist_ok=True)
# 
# results = {
#     'config': CONFIG,
#     'history': history,
#     'final_params': params,
#     'final_solution': u_final,
#     'final_force': force_final,
# }
# 
# filename = f"{save_path}/{CONFIG['problem']}_{CONFIG['discretization']}_{CONFIG['optimizer']}.pkl"
# with open(filename, 'wb') as f:
#     pickle.dump(results, f)
# print(f"Results saved to {filename}")

## Quick Examples Runner

Run predefined examples from the paper:

In [ ]:
# Quick runner for paper examples
def run_paper_example(example_number: str, max_iter: int = 500):
    """Run a specific example from the paper."""
    
    configs = {
        '3.1': {
            'problem': 'poisson-1d-scalar',
            'solver': 'poisson',
            'discretization': 'fd',
            'use_neural_network': False,
            'nx': 50,
            'learning_rate': 0.1,
        },
        '3.2': {
            'problem': 'poisson-1d-vector',
            'solver': 'poisson',
            'discretization': 'fd',
            'use_neural_network': False,
            'nx': 50,
            'learning_rate': 0.01,
            'regularization': 0.099,
        },
        '3.3': {
            'problem': 'heat-1d',
            'solver': 'heat',
            'discretization': 'fem',
            'use_neural_network': True,
            'nx': 32,
            'nt': 32,
            'nn_layers': [256, 256],
            'learning_rate': 1e-3,
        },
    }
    
    if example_number not in configs:
        print(f"Example {example_number} not found. Available: {list(configs.keys())}")
        return
    
    print(f"\nRunning Example {example_number} from the paper")
    print("="*60)
    
    # Update global CONFIG
    CONFIG.update(configs[example_number])
    CONFIG['max_iterations'] = max_iter
    
    # Run the example
    print(f"Configuration: {configs[example_number]}")
    print(f"\nThis will run {max_iter} iterations...")
    
    return CONFIG

# Example usage:
# run_paper_example('3.3', max_iter=1000)
# Then re-run the cells above to execute with this configuration

print("Available paper examples:")
print("  3.1: 1D Poisson with scalar force")
print("  3.2: 1D Poisson with vector force")
print("  3.3: 1+1D Heat equation with NN force")
print("\nCall run_paper_example('3.X') to configure, then run the training cells above.")

## Summary

This notebook provides a complete framework for running PDE-constrained optimization examples with:

1. **Modular Components**: Easy switching between problems, solvers, and discretizations
2. **GPU Support**: Can run on Google Colab with free GPU access
3. **Neural Network Integration**: Optional NN surrogates for force/parameter estimation  
4. **Comprehensive Visualization**: Loss history, solutions, errors
5. **Batch Testing**: Compare different methods easily

### To run on Google Colab with GPU:

1. Upload this notebook and the module files to Colab
2. Change runtime to GPU
3. Run the setup cells
4. Configure your problem in the CONFIG dictionary
5. Execute the training

### For VSCode integration:

Follow the SSH setup instructions in the first cells to connect VSCode to your Colab instance.

### Next steps:

- Try different problems from the paper
- Compare discretization methods
- Experiment with neural network architectures
- Add new PDEs to the problems module